In [1]:
import json
import time
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder, SplineTransformer,
                                   StandardScaler)

import scripts.feature_engineering as feature
from scripts.evaluate import CLIP, calculate_metrics, rmse

c:\Users\Admin\Desktop\Projects\ml-tabular-pfs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('data/gold/train.csv')
df_cat = pd.read_csv('data/landing/item_categories.csv')
df_items = pd.read_csv('data/landing/items.csv')
df_shops = pd.read_csv('data/landing/shops.csv')
df_sales = pd.read_csv('data/landing/sales_train.csv')

df = feature.add_categories(df, df_cat, df_items)
df = feature.add_cities(df, df_shops)
df = feature.add_target_lags(df, (1, 2, 3, 4, 5, 6, 12))
df = feature.add_lag_stats(df)
df = feature.add_mean_lags(df)
df = feature.add_age_features(df)
df = feature.add_price_features(df, df_sales, df_items)
df = feature.add_calendar(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10884554 entries, 0 to 10884553
Data columns (total 34 columns):
 #   Column                Dtype  
---  ------                -----  
 0   date_block_num        int64  
 1   shop_id               int64  
 2   item_id               int64  
 3   target                float64
 4   item_category_id      uint8  
 5   category_type_id      uint8  
 6   item_type_id          uint8  
 7   city_id               uint8  
 8   target_lag_1          float32
 9   target_lag_2          float32
 10  target_lag_3          float32
 11  target_lag_4          float32
 12  target_lag_5          float32
 13  target_lag_6          float32
 14  target_lag_12         float32
 15  roll3                 float32
 16  roll6                 float32
 17  diff_1_2              float32
 18  item_mean_lag_1       float32
 19  shop_mean_lag_1       float32
 20  cat_mean_lag_1        float32
 21  shop_cat_mean_lag_1   float32
 22  city_item_mean_lag_1  float32
 23  pair_age        

In [3]:
LOG1P_COLS = [
    'target_lag_1', 'target_lag_2', 'target_lag_3', 'target_lag_4', 'target_lag_5', 'target_lag_6',
    'target_lag_12', 'roll3', 'roll6',
    'item_mean_lag_1', 'shop_mean_lag_1', 'cat_mean_lag_1', 'shop_cat_mean_lag_1', 'city_item_mean_lag_1',
    'pair_age', 'pair_since_last', 'item_age', 'item_price_last',
]
LOG_COLS = ['item_price_ratio', 'price_rel_cat']
PLAIN_COLS = ['diff_1_2', 'month_sin', 'month_cos', 'item_is_new']
ONEHOT_COLS = ['shop_id', 'item_category_id']
LINEAR_FEATURES = LOG1P_COLS + LOG_COLS + PLAIN_COLS + ONEHOT_COLS

# слабые по SHAP (lgbm/model.ipynb), кандидаты на выброс
WEAK = ['target_lag_3', 'target_lag_4', 'target_lag_5', 'target_lag_6',
        'item_price_last', 'item_price_ratio', 'price_rel_cat', 'month_sin', 'month_cos']
# самые сильные: на них сплайны
SPLINE_COLS = ['target_lag_1', 'item_mean_lag_1', 'shop_cat_mean_lag_1', 'item_age']

TRAIN_SAMPLE = 2_000_000  # финальная модель
TUNE_SAMPLE = 1_000_000   # trials: подбор в 2 раза быстрее, ранжирование параметров почти не меняется

m = df['date_block_num']
train, valid = df[(m >= 12) & (m <= 31)], df[m == 32]
train = train.sample(min(TRAIN_SAMPLE, len(train)), random_state=42)
tune = train.sample(min(TUNE_SAMPLE, len(train)), random_state=42)
y_valid = valid['target'].to_numpy()


def safe_log(x):
    return np.log(np.clip(x, 1e-3, None))


def numeric(func=None):
    """func -> NaN в 0 с флагом пропуска -> стандартизация."""
    steps = [FunctionTransformer(func, feature_names_out='one-to-one')] if func else []
    return make_pipeline(
        *steps,
        SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True),
        StandardScaler(),
    )


def build_prep(use_log: bool, n_knots: int, min_freq: int, drop_weak: bool) -> ColumnTransformer:
    """Подготовка с параметрами; n_knots = 0 - без сплайнов."""
    keep = (lambda cols: [c for c in cols if c not in WEAK]) if drop_weak else list
    parts = [
        ('log1p', numeric(np.log1p if use_log else None), keep(LOG1P_COLS)),
        ('log', numeric(safe_log if use_log else None), keep(LOG_COLS)),
        ('plain', numeric(), keep(PLAIN_COLS)),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=min_freq,
                                 dtype=np.float32), ONEHOT_COLS),
    ]
    if n_knots:
        # сплайн-базис поверх тех же признаков: кусочно-кубическая функция вместо прямой
        parts.append(('spline', make_pipeline(
            FunctionTransformer(np.log1p, feature_names_out='one-to-one'),
            SimpleImputer(strategy='constant', fill_value=0.0),
            SplineTransformer(n_knots=n_knots, degree=3, extrapolation='constant'),
        ), SPLINE_COLS))
    return ColumnTransformer([p for p in parts if p[2]], sparse_threshold=1.0)

In [4]:
STUDY_NAME = 'poisson_v1'
OPTUNA_DIR = Path('data/optuna')
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
MAX_ITER = 1000


def suggest_params(trial: optuna.Trial) -> dict[str, Any]:
    use_splines = trial.suggest_categorical('use_splines', [False, True])
    return {
        'alpha': trial.suggest_float('alpha', 1e-7, 1e-1, log=True),
        'use_log': trial.suggest_categorical('use_log', [True, False]),
        'n_knots': trial.suggest_int('n_knots', 3, 8) if use_splines else 0,
        'min_freq': trial.suggest_int('min_freq', 1, 10_000, log=True),
        'drop_weak': trial.suggest_categorical('drop_weak', [False, True]),
    }


def fit_glm(params: dict[str, Any], data: pd.DataFrame) -> tuple[Any, bool]:
    """Подготовка + PoissonRegressor; возвращает пайплайн и флаг сходимости."""
    prep = build_prep(params['use_log'], params['n_knots'], params['min_freq'], params['drop_weak'])
    model = make_pipeline(prep, PoissonRegressor(alpha=params['alpha'], max_iter=MAX_ITER))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(data[LINEAR_FEATURES], data['target'])
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    return model, converged


def objective(trial: optuna.Trial) -> float:
    params = suggest_params(trial)
    t = time.perf_counter()
    model, converged = fit_glm(params, tune)
    trial.set_user_attr('converged', converged)
    trial.set_user_attr('n_iter', int(model[-1].n_iter_))
    trial.set_user_attr('n_features', int(model[-1].coef_.size))
    trial.set_user_attr('fit_s', round(time.perf_counter() - t, 1))
    return rmse(y_valid, np.clip(model.predict(valid[LINEAR_FEATURES]), *CLIP))

In [5]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f'sqlite:///{OPTUNA_DIR / "poisson.db"}',
    load_if_exists=True,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
# точка старта: конфигурация из poisson/model.ipynb
if len(study.trials) == 0:
    study.enqueue_trial({'alpha': 1e-4, 'use_log': True, 'use_splines': False, 'min_freq': 1, 'drop_weak': False})

study.optimize(objective, n_trials=30, show_progress_bar=True)

[I 2026-10-31 01:08:09,126] A new study created in RDB with name: poisson_v1
Best trial: 0. Best value: 0.928549:   3%|▎         | 1/30 [00:47<22:47, 47.14s/it]

[I 2026-10-31 01:08:56,277] Trial 0 finished with value: 0.9285485317341092 and parameters: {'use_splines': False, 'alpha': 0.0001, 'use_log': True, 'min_freq': 1, 'drop_weak': False}. Best is trial 0 with value: 0.9285485317341092.


Best trial: 1. Best value: 0.919959:   7%|▋         | 2/30 [01:26<19:58, 42.82s/it]

[I 2026-10-31 01:09:36,079] Trial 1 finished with value: 0.9199594728252232 and parameters: {'use_splines': True, 'alpha': 0.002465832945854913, 'use_log': True, 'n_knots': 3, 'min_freq': 1, 'drop_weak': False}. Best is trial 1 with value: 0.9199594728252232.


Best trial: 1. Best value: 0.919959:  10%|█         | 3/30 [01:32<11:35, 25.77s/it]

[I 2026-10-31 01:09:41,569] Trial 2 finished with value: 0.9391783901112492 and parameters: {'use_splines': False, 'alpha': 0.06598711072054077, 'use_log': True, 'min_freq': 3, 'drop_weak': True}. Best is trial 1 with value: 0.9199594728252232.


... (50 outputs left out)

Best trial: 15. Best value: 0.898936:  97%|█████████▋| 29/30 [30:16<01:21, 81.60s/it]

[I 2026-10-31 01:38:26,038] Trial 28 finished with value: 0.9046567373973127 and parameters: {'use_splines': True, 'alpha': 2.6525229734303336e-06, 'use_log': True, 'n_knots': 7, 'min_freq': 2, 'drop_weak': False}. Best is trial 15 with value: 0.8989363013406567.


Best trial: 29. Best value: 0.896498: 100%|██████████| 30/30 [31:55<00:00, 63.85s/it]

[I 2026-10-31 01:40:04,655] Trial 29 finished with value: 0.8964983878850962 and parameters: {'use_splines': True, 'alpha': 1.4429937238040738e-05, 'use_log': False, 'n_knots': 6, 'min_freq': 5, 'drop_weak': True}. Best is trial 29 with value: 0.8964983878850962.


In [6]:
print('лучший RMSE:', round(study.best_value, 4))
print(study.best_trial.user_attrs)
study.best_params

лучший RMSE: 0.8965
{'converged': True, 'fit_s': 97.9, 'n_features': 190, 'n_iter': 803}


{'use_splines': True,
 'alpha': 1.4429937238040738e-05,
 'use_log': False,
 'n_knots': 6,
 'min_freq': 5,
 'drop_weak': True}

In [7]:
trials = study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs', 'state'))
trials.sort_values('value').head(10)

,number,value,params_alpha,params_drop_weak,params_min_freq,params_n_knots,params_use_log,params_use_splines,user_attrs_converged,user_attrs_fit_s,user_attrs_n_features,user_attrs_n_iter,state
29,29,0.896498,1.442994e-05,True,5,6.0,False,True,True,97.9,190,803,COMPLETE
15,15,0.898936,7.577236e-07,True,1,8.0,True,True,True,69.7,198,557,COMPLETE
6,6,0.899022,3.570096e-06,True,112,6.0,True,True,True,70.2,181,549,COMPLETE
14,14,0.899164,2.391160e-07,True,2,6.0,True,True,True,69.4,190,552,COMPLETE
21,21,0.899500,2.248354e-07,True,1,6.0,True,True,True,73.5,190,572,COMPLETE
12,12,0.899551,3.143573e-07,True,23,5.0,True,True,True,94.5,181,768,COMPLETE
13,13,0.899627,4.702680e-07,True,48,5.0,True,True,True,66.9,181,531,COMPLETE
23,23,0.899647,9.415724e-06,True,1,6.0,True,True,True,71.1,190,569,COMPLETE
17,17,0.901218,1.796799e-06,False,7,8.0,False,True,False,151.9,214,1000,COMPLETE
24,24,0.901886,4.983522e-06,True,20,4.0,True,True,True,88.9,178,709,COMPLETE


In [8]:
optuna.visualization.plot_optimization_history(study).show()
optuna.visualization.plot_param_importances(study).show()

In [9]:
optuna.visualization.plot_slice(study, params=['alpha', 'min_freq', 'n_knots']).show()

In [10]:
best = study.best_params
best_params = {
    'alpha': best['alpha'], 'use_log': best['use_log'], 'min_freq': best['min_freq'],
    'drop_weak': best['drop_weak'], 'n_knots': best['n_knots'] if best['use_splines'] else 0,
}
model, converged = fit_glm(best_params, train)
print('сошлась:', converged, '| итераций:', model[-1].n_iter_)

res = calculate_metrics(
    model,
    splits={'train': (train[LINEAR_FEATURES], train['target']), 'valid': (valid[LINEAR_FEATURES], valid['target'])},
    model_name='Poisson (Optuna)',
    baselines={'zeros': np.zeros(len(valid)), 'naive': valid['target_lag_1'].fillna(0)},
)
res.round(4)

сошлась: True | итераций: 769


,rmse,mae,bias,r2,rmse_nonzero
train,0.8915,0.3162,-0.0010,0.4442,2.1920
valid,0.8977,0.3099,0.0112,0.4086,2.2565
valid: zeros,1.1982,0.2701,-0.2701,-0.0536,3.2177
valid: naive,1.0766,0.3202,0.0106,0.1494,2.6185


In [11]:
# эффекты числовых признаков: прогноз умножается на multiplier при росте признака на 1 ст. отклонение
coef = pd.Series(model[-1].coef_, index=model[0].get_feature_names_out())
effect = pd.DataFrame({'coef': coef, 'multiplier': np.exp(coef)})
num = effect[~effect.index.str.startswith(('onehot__', 'spline__'))]
num.reindex(num['coef'].abs().sort_values(ascending=False).index).head(15).round(3)

,coef,multiplier
log1p__item_age,-0.523,0.593
log1p__missingindicator_city_item_mean_lag_1,-0.318,0.727
log1p__pair_since_last,-0.308,0.735
log1p__pair_age,0.262,1.300
log1p__shop_mean_lag_1,-0.207,0.813
log1p__missingindicator_item_mean_lag_1,0.182,1.199
log1p__item_mean_lag_1,0.154,1.167
log1p__missingindicator_pair_age,-0.136,0.873
log1p__missingindicator_pair_since_last,-0.136,0.873
log1p__shop_cat_mean_lag_1,0.132,1.141


In [12]:
(OPTUNA_DIR / f'{STUDY_NAME}_best_params.json').write_text(json.dumps(best_params, indent=2))
best_params

{'alpha': 1.4429937238040738e-05,
 'use_log': False,
 'min_freq': 5,
 'drop_weak': True,
 'n_knots': 6}